# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Answer: Hidden states are dense floating-point tensors, not compact text or token IDs. Storage grows with `sequence_length * captured_layers * hidden_size * dtype_bytes`; for BF16 this is 2 bytes per element, so a few thousand 2048-token samples with multiple captured layers can reach hundreds of GB.

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Measured validation result from the Nebius H100 run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.997` |
| `val/full_acc_0_epoch` | `0.429` |
| `val/cond_acc_0_epoch` | `0.429` |
| `val/loss_1_epoch` | `4.252` |
| `val/full_acc_1_epoch` | `0.154` |
| `val/cond_acc_1_epoch` | `0.354` |
| `val/loss_2_epoch` | `4.993` |
| `val/full_acc_2_epoch` | `0.053` |
| `val/cond_acc_2_epoch` | `0.331` |
| `val/loss_epoch` | `12.241` |

Evidence: `docs/evidence/training/val_metrics.json`.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

Answers:

1. `full_acc` measures whether the speculative sequence remains correct through a given draft position. `cond_acc` measures accuracy at that position conditioned on the previous drafted positions already being correct.
2. Later positions are harder because the draft head predicts farther ahead with less verifier computation, and any earlier mismatch changes the context for subsequent predictions. That is why the measured full accuracy drops from `0.429` at position 0 to `0.154` and `0.053` at later positions.
3. If first-position accuracy is very low, debug data generation before changing training hyperparameters: verify the model/tokenizer pair, assistant masks, hidden-state length versus token length, vLLM compatibility, and stale temporary hidden-state files.

---

## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

Answers:

1. FP8 dynamic quantization is useful on H100 because Hopper GPUs have fast FP8 execution, and FP8 reduces memory bandwidth pressure for linear layers. In the assignment benchmark, FP8 improved output throughput from `1168.59` to `1701.18` tok/s.
2. `lm_head` directly produces vocabulary logits, so quantization noise there can disproportionately affect generation quality and verifier/draft agreement. Keeping it unquantized protects the final probability distribution.
3. Quantization can slightly shift verifier logits. A draft head trained against the BF16 verifier may therefore see a different acceptance pattern when paired with the FP8 verifier, so FP8 + speculative decoding must be benchmarked separately.

---

## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Assignment benchmark profile used for the passing H100 run:

```bash
export BENCH_NUM_PROMPTS=80
export BENCH_CONCURRENCY=8
export MAX_MODEL_LEN=2048
export GPU_MEMORY_UTILIZATION=0.95
export MAX_NUM_SEQS=64
export MAX_NUM_BATCHED_TOKENS=8192
export BENCH_OUTPUT_LEN=256
export BENCH_IGNORE_EOS=1
unset VLLM_SERVE_EXTRA_ARGS
```

The benchmark command used by `scripts/bench_one.sh` is still `vllm bench serve` with the assignment dataset and tokenizer:

```bash
vllm bench serve \
    --backend openai-chat \
    --base-url http://localhost:8000/v1 \
    --endpoint /chat/completions \
    --dataset-name hf \
    --dataset-path philschmid/mt-bench \
    --max-concurrency 8 \
    --num-prompts 80 \
    --hf-output-len 256 \
    --ignore-eos \
    --temperature 0 \
    --seed 42 \
    --tokenizer Qwen/Qwen3-8B
```

Important serving note: `--enforce-eager` was not used for scoring. It is only a hidden-state-generation workaround; when enabled for serving it disables CUDA graph execution and can make vLLM much slower.

Measured assignment benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Completed / Failed |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `17.53` | `4.56` | `1168.59` | `1551.91` | `40.57` | `6.71` | `80 / 0` |
| Speculative decoding | `16.05` | `4.99` | `1276.33` | `1695.00` | `129.29` | `5.45` | `80 / 0` |
| FP8 quantization | `12.04` | `6.65` | `1701.18` | `2259.22` | `32.05` | `4.59` | `80 / 0` |
| FP8 + speculative decoding | `10.91` | `7.33` | `1877.15` | `2492.90` | `56.11` | `3.90` | `80 / 0` |

Measured speculative settings:

| Configuration | Configured draft tokens | Acceptance metrics | Mean TPOT, ms | Output tok/s |
| --- | ---: | --- | ---: | ---: |
| Speculative decoding | `2` | Not emitted in saved benchmark JSON | `5.45` | `1276.33` |
| FP8 + speculative decoding | `1` | Not emitted in saved benchmark JSON | `3.90` | `1877.15` |

The higher-load tuning profile is also preserved in `docs/evidence/benchmarks/*score*.json`: baseline `4297.31`, speculative `4315.99`, FP8 `5895.88`, and FP8 + speculative `5809.51` output tok/s at concurrency 32 and 256 prompts.

Evidence: `docs/evidence/benchmarks/*c8_p80*.json` and `docs/evidence/benchmarks/*score*.json`.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

Answers:

1. Speculative decoding can improve throughput because accepted draft tokens let a single verifier step validate and emit more than one output token. Acceptance does not need to be 100%; it only needs to save enough verifier work to offset the draft-model overhead.
2. For the measured assignment profile, the best tested settings are `SPEC_TOKENS=2` for BF16 speculative serving and `FP8_SPEC_TOKENS=1` for FP8 + speculative serving. The saved benchmark JSONs do not include acceptance rate or acceptance length, so the choice is based on measured TPOT and throughput: BF16 speculative improved TPOT from `6.71` to `5.45` ms and reached `1276.33` tok/s, while FP8 + speculative reached the best assignment result, `1877.15` tok/s with `3.90` ms TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Measured result | Points |
| --- | ---: | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | `1276.33 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | `1701.18 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | `1877.15 tok/s` | 15 |
| **Total** |  |  | **50** |

Baseline assignment benchmark result:
```
============ Serving Benchmark Result ============
Successful requests:                     80
Failed requests:                         0
Maximum request concurrency:             8
Benchmark duration (s):                  17.53
Total input tokens:                      6718
Total generated tokens:                  20480
Request throughput (req/s):              4.56
Output token throughput (tok/s):         1168.59
Peak output token throughput (tok/s):    1200.00
Peak concurrent requests:                16.00
Total token throughput (tok/s):          1551.91
---------------Time to First Token----------------
Mean TTFT (ms):                          40.57
Median TTFT (ms):                        40.97
P99 TTFT (ms):                           61.28
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          6.71
Median TPOT (ms):                        6.71
P99 TPOT (ms):                           6.78
---------------Inter-token Latency----------------
Mean ITL (ms):                           6.68
Median ITL (ms):                         6.69
P99 ITL (ms):                            7.20
==================================================
```

Speculative decoding assignment benchmark results:
```
============ Serving Benchmark Result ============
Successful requests:                     80
Failed requests:                         0
Maximum request concurrency:             8
Benchmark duration (s):                  16.05
Request throughput (req/s):              4.99
Output token throughput (tok/s):         1276.33
Total token throughput (tok/s):          1695.00
Mean TTFT (ms):                          129.29
Mean TPOT (ms):                          5.45
Configured draft tokens:                 2
Acceptance rate:                         Not emitted in saved benchmark JSON
Acceptance length:                       Not emitted in saved benchmark JSON
==================================================
```

This run used two configured speculative draft tokens. It reduced TPOT from the assignment-profile baseline 6.71 ms to 5.45 ms and increased output throughput from 1168.59 to 1276.33 tok/s, passing the 1250 tok/s scoring threshold.

FP8 quantization assignment benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80
Failed requests:                         0
Maximum request concurrency:             8
Benchmark duration (s):                  12.04
Request throughput (req/s):              6.65
Output token throughput (tok/s):         1701.18
Total token throughput (tok/s):          2259.22
Mean TTFT (ms):                          32.05
Mean TPOT (ms):                          4.59
Acceptance rate:                         N/A
==================================================
```

The FP8 dynamic verifier is faster than BF16 baseline in the assignment run: output throughput increased from 1168.59 to 1701.18 tok/s and TPOT improved from 6.71 ms to 4.59 ms. This passes the 1550 tok/s scoring threshold.

FP8 + speculative decoding assignment benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80
Failed requests:                         0
Maximum request concurrency:             8
Benchmark duration (s):                  10.91
Request throughput (req/s):              7.33
Output token throughput (tok/s):         1877.15
Total token throughput (tok/s):          2492.90
Mean TTFT (ms):                          56.11
Mean TPOT (ms):                          3.90
Configured draft tokens:                 1
Acceptance rate:                         Not emitted in saved benchmark JSON
Acceptance length:                       Not emitted in saved benchmark JSON
==================================================
```

The combined FP8 + speculative configuration used one configured draft token and reached 1877.15 tok/s, passing the 1750 tok/s scoring threshold. This is the fastest assignment-profile result. The higher-load concurrency-32 run is also preserved and shows FP8 alone slightly ahead at that traffic level.

## Final Answers

### Which optimization should be done first?

Train the EAGLE-3 speculative decoding draft head first, then quantize the verifier. The draft head learns from verifier hidden states and token targets, so it should see the original BF16 verifier distribution during training. FP8 dynamic quantization is then applied as a serving optimization and validated by benchmark results.

### Task 1: Hidden-state storage

Hidden states require much more disk space than the text dataset because text is stored as compact strings or token IDs, while hidden-state generation stores dense floating-point tensors. Storage scales approximately as `sequence_length * captured_layers * hidden_size * dtype_bytes`, so thousands of samples with several BF16 hidden-state layers can become hundreds of GB.

### Task 2: Training metrics

`full_acc` measures whether drafted tokens remain correct through a speculative position. `cond_acc` measures correctness at a position conditioned on earlier draft positions already being correct. Accuracy decreases for later positions because the draft head is predicting farther ahead with less verifier computation and earlier mistakes compound. If first-position accuracy is very low, inspect data generation first: model/tokenizer match, assistant mask, hidden-state length versus token length, vLLM version, and stale `/tmp/hidden_states` files.

The measured first-position validation accuracy was `full_acc_0_epoch=0.429` and `cond_acc_0_epoch=0.429`; later full accuracy dropped to `0.154` and `0.053`, which is the expected pattern for farther speculative positions.

### Task 3: Quantization

FP8 dynamic quantization is useful on H100 because Hopper GPUs accelerate FP8 operations and the lower precision reduces memory bandwidth pressure. `lm_head` is kept unquantized because it directly controls final vocabulary logits, so quantization noise there can harm generation quality and speculative acceptance. Quantization can change verifier logits slightly, so the BF16-trained draft head may agree with the FP8 verifier at a different rate; this is why FP8 + speculative decoding is benchmarked separately.

In the measured assignment run, FP8 alone improved output throughput from `1168.59` to `1701.18` tok/s and reduced TPOT from `6.71` to `4.59` ms.

### Task 4: Speculative benchmark interpretation

Speculative decoding can improve throughput even without 100% acceptance because accepted draft tokens let one verifier step validate and emit more than one output token. The draft model is cheaper than the verifier, so partial acceptance can still reduce average verifier work per output token.

In the assignment run, BF16 speculative serving used two configured draft tokens and improved output throughput from `1168.59` to `1276.33` tok/s while reducing TPOT from `6.71` to `5.45` ms. FP8 + speculative serving used one configured draft token and reached `1877.15` tok/s with `3.90` ms TPOT. The saved benchmark JSONs did not emit acceptance rate or acceptance length, so the optimal draft-token claim is based on measured throughput and TPOT. The higher-load c32 evidence is also preserved; there FP8 alone was slightly faster than FP8 + speculative.